<a href="https://colab.research.google.com/github/wanjir0/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/agent_cleanup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import numpy as np, pandas as pd

rng = np.random.default_rng(44)

n = 600

agents = pd.DataFrame({

"agent_id": [f"AG{i:03d}" for i in range(1, 21)],
"county": rng.choice(["Nairobi", "Kiambu", "Nakuru", "Mombasa", "Kisumu"], 20),

"onboarded": pd.date_range("2024-01-01", periods=20, freq="MS")

.strftime("%Y-%m-%d"),

})

txns = pd.DataFrame({

"txn_id": [f"TX{i:05d}" for i in range(1, n + 1)],

"agent_id": rng.choice(agents["agent_id"], n),

"txn_type": rng.choice(["Deposit", "withdrawal", "DEPOSIT ",

"Withdrawal", "deposit"], n),

"amount": rng.lognormal(7.5, 0.8, n).round(0).astype(str),
"txn_date": (pd.to_datetime("2026-03-01")

+ pd.to_timedelta(rng.integers(0, 60, n), unit="D"))

.strftime("%Y-%m-%d"),

})

idx = rng.choice(n, 30, replace=False)

txns.loc[idx[:15], "amount"] = np.nan

txns.loc[idx[15:20], "amount"] = "N/A"

txns.loc[idx[20:25], "amount"] = "250000"

txns.loc[idx[25:], "txn_date"] = txns.loc[idx[25:], "txn_date"].str.replace("-", "/")

txns = pd.concat([txns, txns.sample(12, random_state=1)], ignore_index=True)

txns.loc[rng.choice(len(txns), 8, replace=False), "agent_id"] = "AG099"

agents.to_csv("agents.csv", index=False)

txns.to_csv("agent_transactions.csv", index=False)

In [31]:
import pandas as pd
import numpy as np

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

print("Transactions:", txns.shape, "| Agents:", agents.shape)
print(txns.dtypes)
print(txns.isna().sum())

# Second, text-only load. pandas silently turns "N/A" into NaN in the first load,
# so reading everything as text lets us tell blank values from the literal "N/A".
raw = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)

amount_num = pd.to_numeric(raw["amount"], errors="coerce")
slash_dates = raw["txn_date"].str.contains("/", regex=False)

issues = [
    ("Duplicate transaction IDs (surplus copies)", "Uniqueness", "txn_id",
     raw["txn_id"].duplicated().sum()),
    ("Fully identical duplicate rows", "Uniqueness", "all columns",
     raw.duplicated().sum()),
    ("txn_type not exactly 'Deposit' or 'Withdrawal'", "Consistency", "txn_type",
     (~raw["txn_type"].isin(["Deposit", "Withdrawal"])).sum()),
    ("txn_type with leading/trailing whitespace", "Consistency", "txn_type",
     (raw["txn_type"] != raw["txn_type"].str.strip()).sum()),
    ("Amount blank (missing)", "Completeness", "amount",
     (raw["amount"].str.strip() == "").sum()),
    ("Amount is the text 'N/A'", "Completeness", "amount",
     (raw["amount"].str.strip().str.upper() == "N/A").sum()),
    ("Amount of exactly KES 250,000 (suspicious round value)", "Accuracy", "amount",
     (amount_num == 250000).sum()),
    ("Dates written with '/' instead of '-'", "Validity", "txn_date",
     slash_dates.sum()),
    ("agent_id not in the agent register", "Integrity", "agent_id",
     (~raw["agent_id"].isin(agents["agent_id"])).sum()),
]

audit = pd.DataFrame(issues, columns=["issue", "dimension", "column", "rows_affected"])
audit["rows_affected"] = audit["rows_affected"].astype(int)
audit["pct_of_rows"] = (audit["rows_affected"] / len(raw) * 100).round(1)
audit

Transactions: (612, 5) | Agents: (20, 3)
txn_id       object
agent_id     object
txn_type     object
amount      float64
txn_date     object
dtype: object
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0
dtype: int64


,issue,dimension,column,rows_affected,pct_of_rows
0,Duplicate transaction IDs (surplus copies),Uniqueness,txn_id,12,2.0
1,Fully identical duplicate rows,Uniqueness,all columns,11,1.8
2,txn_type not exactly 'Deposit' or 'Withdrawal',Consistency,txn_type,384,62.7
3,txn_type with leading/trailing whitespace,Consistency,txn_type,111,18.1
4,Amount blank (missing),Completeness,amount,15,2.5
5,Amount is the text 'N/A',Completeness,amount,7,1.1
6,"Amount of exactly KES 250,000 (suspicious roun...",Accuracy,amount,5,0.8
7,Dates written with '/' instead of '-',Validity,txn_date,5,0.8
8,agent_id not in the agent register,Integrity,agent_id,8,1.3


### Data-quality audit
 The extract has 612 rows, but the generator describes 600 transactions, so some rows are surplus. The agent register has 20 agents, with no missing values.

**Method note:** `amount` loads as a float in pandas because "N/A" is read as missing automatically. To see the real problems I also loaded the file as text (`dtype=str, keep_default_na=False`) and counted issues from that.

| Dimension | Issue | Column | Rows |
|---|---|---|---|
| Uniqueness | Surplus copies of an existing `txn_id` | txn_id | 12 |
| Consistency | `txn_type` has 5 spellings instead of 2 (case and trailing-space variants) | txn_type | 384 |
| Completeness | Amount missing (15 blank, 7 as text "N/A") | amount | 22 |
| Accuracy | Round amount of KES 250,000, far above the typical range (to be investigated in the outlier step) | amount | 5 |
| Validity | Date format varies ("/" instead of "-") | txn_date | 5 |
| Integrity | `agent_id` AG099 does not exist in the register | agent_id | 8 |

Counts overlap (for example, a row can have both a bad `txn_type` and a missing amount), so the rows column must not be added up.

In [32]:
before = len(txns)

# Full-row check (what most people try first)
print("Exact duplicate rows:     ", txns.duplicated().sum())
print("Duplicate transaction IDs:", txns["txn_id"].duplicated().sum())

# Look at the pair that the full-row check misses
conflict = txns[txns["txn_id"].duplicated(keep=False)].groupby("txn_id").filter(
    lambda g: len(g.drop_duplicates()) > 1)
display(conflict)

# Keep the copy whose agent_id exists in the register, then drop the rest
txns_dedup = (txns
    .assign(_valid_agent=txns["agent_id"].isin(agents["agent_id"]))
    .sort_values("_valid_agent", ascending=False, kind="stable")
    .drop_duplicates(subset="txn_id", keep="first")
    .drop(columns="_valid_agent")
    .sort_values("txn_id")
    .reset_index(drop=True))

removed = before - len(txns_dedup)
print(f"\nRows before: {before} | after: {len(txns_dedup)} | removed: {removed}")
print("Duplicate txn_ids remaining:", txns_dedup["txn_id"].duplicated().sum())

Exact duplicate rows:      11
Duplicate transaction IDs: 12


,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03



Rows before: 612 | after: 600 | removed: 12
Duplicate txn_ids remaining: 0


### Uniqueness: duplicate transactions
`txn_id` is the unique key, so duplicates were identified on `txn_id`, not on whole rows.

- **12 duplicate transactions were removed** (612 rows reduced to 600).
- An exact-row check finds only 11. The twelfth, TX00405, has two copies that differ only in `agent_id` (AG014 vs the unregistered AG099), so it is a duplicate with a corrupted copy.
- For that pair I kept the copy with the valid, registered agent (AG014) and removed the AG099 copy. Keeping the first row at random could have kept the wrong agent.
- Check: 600 rows now match the 600 transaction IDs (TX00001 to TX00600), with no repeated `txn_id`.

In [33]:
clean = txns_dedup.copy()

print("Before:\n", clean["txn_type"].value_counts(dropna=False))
print("Missing txn_type:", clean["txn_type"].isna().sum())   # check isna() as well as bad strings

type_map = {"deposit": "Deposit", "withdrawal": "Withdrawal"}

clean["txn_type"] = (clean["txn_type"]
                     .str.strip()          # removes the trailing space in "DEPOSIT "
                     .str.lower()          # removes case differences
                     .map(type_map))       # anything unexpected becomes NaN

# Safety checks: nothing missing, nothing unmapped, exactly two values
print("\nUnmapped or missing after cleaning:", clean["txn_type"].isna().sum())
assert clean["txn_type"].isna().sum() == 0
assert set(clean["txn_type"].unique()) == {"Deposit", "Withdrawal"}

print("\nAfter:\n", clean["txn_type"].value_counts())

Before:
 txn_type
deposit       138
withdrawal    128
Deposit       125
DEPOSIT       110
Withdrawal     99
Name: count, dtype: int64
Missing txn_type: 0

Unmapped or missing after cleaning: 0

After:
 txn_type
Deposit       373
Withdrawal    227
Name: count, dtype: int64


### Consistency: `txn_type`
The column had five spellings of two categories: "deposit", "Deposit", "DEPOSIT " (with a trailing space), "withdrawal" and "Withdrawal".

**Fix:** strip whitespace, convert to lower case, then map to the two approved labels, "Deposit" and "Withdrawal".

**Checks:**
- `txn_type` had no missing values (`isna()` count 0), so no rows were lost.
- Every value mapped to one of the two labels (0 unmapped), and an `assert` confirms exactly two values remain.
- Result: 373 Deposits and 227 Withdrawals, adding up to 600 rows.

In [34]:
# --- amount -> numeric ---
print("amount dtype before:", clean["amount"].dtype)

clean["amount"] = pd.to_numeric(
    clean["amount"].astype(str).str.replace(",", "", regex=False).str.strip(),
    errors="coerce")          # "N/A", "nan", blanks -> NaN instead of crashing

print("amount dtype after: ", clean["amount"].dtype)
print("Non-numeric / missing amounts:", clean["amount"].isna().sum())

# --- txn_date -> datetime ---
slash_before = clean["txn_date"].str.contains("/", regex=False).sum()
print("\nDates using '/':", slash_before)

naive = pd.to_datetime(clean["txn_date"], errors="coerce")
print("Dates lost by a naive pd.to_datetime:", naive.isna().sum())

# Standardise the separator first, then parse with an explicit format
clean["txn_date"] = pd.to_datetime(
    clean["txn_date"].str.replace("/", "-", regex=False),
    format="%Y-%m-%d", errors="coerce")

print("Unparseable dates after fix:", clean["txn_date"].isna().sum())
print("Date range:", clean["txn_date"].min().date(), "to", clean["txn_date"].max().date())
clean.dtypes

amount dtype before: float64
amount dtype after:  float64
Non-numeric / missing amounts: 20

Dates using '/': 5
Dates lost by a naive pd.to_datetime: 5
Unparseable dates after fix: 0
Date range: 2026-03-01 to 2026-04-29


,0
txn_id,object
agent_id,object
txn_type,object
amount,float64
txn_date,datetime64[ns]


### Validity and types
**`amount`:** converted with `pd.to_numeric(errors="coerce")` after removing commas and whitespace. Values that cannot be read as numbers ("N/A", blanks) become missing (`NaN`) instead of stopping the code. The result is numeric with **20 missing values** (from 15 blank and 5 "N/A" after de-duplication). No negative or zero amounts exist. Missing values are handled in question 14.

**`txn_date`:** 5 dates were written as `2026/04/27` instead of `2026-04-27`. A plain `pd.to_datetime` silently turned those 5 into `NaT` (missing). I replaced "/" with "-" and parsed with an explicit `%Y-%m-%d` format. The format is year-month-day, so there is no day/month ambiguity.

**Checks:** 0 dates unparseable. All transactions fall between 1 March and 29 April 2026, so there are no future or impossible dates.

In [35]:
clean["amount_missing"] = clean["amount"].isna() | (clean["amount"] <= 0)

n_bad = clean["amount_missing"].sum()
print(f"Rows with missing/invalid amount: {n_bad} ({n_bad / len(clean):.1%} of {len(clean)})")

print("\nBy transaction type:")
print(clean.groupby("txn_type")["amount_missing"].agg(["sum", "mean"]).round(3))
print("\nAgents affected:", clean.loc[clean["amount_missing"], "agent_id"].nunique())

median_amt = clean["amount"].median()
print(f"\nMedian valid amount: KES {median_amt:,.0f}")
print(f"Imputing the median would add KES {median_amt * n_bad:,.0f} of invented value "
      f"({median_amt * n_bad / clean['amount'].sum():.1%} of total)")

Rows with missing/invalid amount: 20 (3.3% of 600)

By transaction type:
            sum   mean
txn_type              
Deposit      13  0.035
Withdrawal    7  0.031

Agents affected: 15

Median valid amount: KES 1,776
Imputing the median would add KES 35,530 of invented value (1.3% of total)


### Completeness: missing and invalid amounts
**Problem:** 20 transactions (3.3% of 600) have no usable amount: 15 blank and 5 written as "N/A". No amounts are zero or negative.

**Decision:** keep the rows, set `amount_missing = True` on them, and leave `amount` empty (`NaN`). Amounts are not filled in. Value totals and averages use only rows where `amount_missing` is False. Transaction counts include all 600 transactions, because the transaction happened and has a valid ID, agent and date.

**Why not impute (mean or median)?** This is a financial record. Filling in an amount creates money that was never moved, and any figure a regulator or auditor sees would partly be invented. Amounts are also right-skewed (median about KES 1,777 against a mean of about KES 4,574), so a mean would overstate small transactions and a median would flatten real variation. Imputing the median would add roughly KES 35,500 to the totals.

**Why not drop the rows?** Dropping would undercount the transactions that agents handled, and it would lose evidence of data loss. Missing amounts appear spread across both transaction types and 15 agents, so dropping them would not obviously bias the value results, but there is no need to lose the rows.

**Impact:** at 3.3% of rows, the effect on county totals is small, and the flag lets the business see exactly which transactions are affected.

**Recommendation:** ask the data owner to recover the 20 amounts from the source ledger, then re-run the pipeline.

In [36]:
valid = clean.loc[~clean["amount_missing"], "amount"]

q1, q3 = valid.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f} | Q3 = {q3:,.0f} | IQR = {iqr:,.0f}")
print(f"Fences: lower = {lower:,.0f} | upper = {upper:,.0f}")

clean["is_outlier_iqr"] = (clean["amount"] > upper) | (clean["amount"] < lower)
clean["amount_suspect"] = clean["amount"] == 250000

print("\nIQR outliers:", clean["is_outlier_iqr"].sum())
print("  of which exactly 250,000:", clean["amount_suspect"].sum())
print("  other outliers (6,085 to 28,847):", (clean["is_outlier_iqr"] & ~clean["amount_suspect"]).sum())
print("Next largest amount below 250,000:", f"{valid[valid < 250000].max():,.0f}")

# Investigate the 250,000 rows
display(clean.loc[clean["amount_suspect"], ["txn_id", "agent_id", "txn_type", "amount", "txn_date"]])

# Impact on the numbers
total, total_ex = valid.sum(), valid[valid != 250000].sum()
print(f"\nShare of all transaction value from the 5 rows: {(total - total_ex) / total:.1%}")
print(f"Mean amount with them: {valid.mean():,.0f} | without: {valid[valid != 250000].mean():,.0f}")

Q1 = 1,077 | Q3 = 3,080 | IQR = 2,003
Fences: lower = -1,928 | upper = 6,084

IQR outliers: 40
  of which exactly 250,000: 5
  other outliers (6,085 to 28,847): 35
Next largest amount below 250,000: 28,847


,txn_id,agent_id,txn_type,amount,txn_date
306,TX00307,AG012,Deposit,250000.0,2026-03-02
315,TX00316,AG014,Deposit,250000.0,2026-04-01
424,TX00425,AG005,Withdrawal,250000.0,2026-03-02
463,TX00464,AG009,Withdrawal,250000.0,2026-04-11
594,TX00595,AG016,Deposit,250000.0,2026-03-11



Share of all transaction value from the 5 rows: 47.1%
Mean amount with them: 4,574 | without: 2,440


### Outliers (IQR method)
**Method:** on valid amounts, Q1 = KES 1,077 and Q3 = KES 3,080, so IQR = 2,003. Amounts above Q3 + 1.5 × IQR = **KES 6,085** are flagged (the lower fence is negative, so nothing is flagged low). **40 transactions** are flagged in `is_outlier_iqr`.

**Two kinds of flag:**
- **35 ordinary high values** (KES 6,158 to 28,847). Transaction amounts are right-skewed, so a long upper tail is expected. These look like real large transactions and are kept without change.
- **5 values of exactly KES 250,000** (`amount_suspect = True`), which I investigated.

**Investigation of the KES 250,000 rows:**
- Exactly the same round number five times, in a column of amounts that otherwise vary in unit shillings. Real amounts do not repeat that precisely.
- Almost nine times larger than the next largest amount (KES 28,847).
- Spread across 5 different agents, both transaction types and 6 weeks, so it does not point to a single agent or day.
- The 5 rows are 0.8% of transactions but 47.1% of all transaction value, and they move the mean amount from about KES 2,440 to KES 4,574. They would dominate any county total.
- 250,000 is also close to the per-transaction limit that mobile-money providers in Kenya apply (as far as I know, M-Pesa's cap is KES 250,000), so these could be limit or default values from the legacy system rather than real amounts. This needs to be confirmed against the provider's rules.

**Recommendation:** do not delete the rows and do not trust them as they stand. Keep them in the data with the `amount_suspect` flag, and verify the five transactions against the source ledger and agent float records. Until they are verified, report county totals and averages **both with and without** them, and use the "without" figures for headline averages. If the ledger confirms the amounts, clear the flag. If they are system defaults or errors, correct them from the ledger, or treat them like missing amounts.

In [37]:
agents["onboarded"] = pd.to_datetime(agents["onboarded"], format="%Y-%m-%d")

# Register must have one row per agent, or the merge would multiply transactions
assert agents["agent_id"].is_unique

rows_before = len(clean)
merged = clean.merge(agents, on="agent_id", how="left",
                     validate="many_to_one", indicator=True)
assert len(merged) == rows_before          # a left join must not add or lose rows

merged["agent_unregistered"] = merged["_merge"] == "left_only"
merged = merged.drop(columns="_merge")

unknown = merged[merged["agent_unregistered"]]
print("Transactions:", len(merged))
print("Unknown-agent transactions:", len(unknown), f"({len(unknown) / len(merged):.1%})")
print("Unknown agent IDs:", unknown["agent_id"].unique())
print(f"Their value: KES {unknown['amount'].sum():,.0f} "
      f"({unknown['amount'].sum() / merged['amount'].sum():.1%} of total)")

# Keep them, but label them clearly instead of leaving county empty
merged["county"] = merged["county"].fillna("Unknown (unregistered agent)")
display(unknown[["txn_id", "agent_id", "txn_type", "amount"]])

Transactions: 600
Unknown-agent transactions: 7 (1.2%)
Unknown agent IDs: ['AG099']
Their value: KES 27,156 (1.0% of total)


,txn_id,agent_id,txn_type,amount
55,TX00056,AG099,Deposit,8507.0
68,TX00069,AG099,Withdrawal,1877.0
77,TX00078,AG099,Deposit,1770.0
237,TX00238,AG099,Deposit,4944.0
267,TX00268,AG099,Deposit,3110.0
444,TX00445,AG099,Deposit,5309.0
557,TX00558,AG099,Deposit,1639.0


### Merge with the agent register
Transactions were joined to `agents.csv` on `agent_id` with a left join (`validate="many_to_one"`), so every transaction is kept and each is matched to one agent. The row count stayed at 600.

**Unknown agents:** **7 transactions (1.2%)** reference agent **AG099**, who is not in the register. Together they are worth KES 27,156 (about 1.0% of total value). This was one of 8 AG099 rows in the raw file. The 8th was a duplicate copy of TX00405, which was removed in the de-duplication step, so its original with the valid agent AG014 was kept.

**How I handled them:** I did not drop them. The transactions are otherwise valid, with a real ID, date, type and amount. I set `agent_unregistered = True` and gave them the county label "Unknown (unregistered agent)". They are shown on their own line in the county summary so they are not mixed into a real county, and county rankings are not distorted.

**Recommendation:** ask the agent-management team whether AG099 is a new, test or deactivated agent, or a keying error for an existing ID. If it is a real agent, add them to the register. If it is an error, correct the IDs from source records. This is also a potential control gap: transactions were processed for an agent who has no register entry.

In [38]:
merged["analysis_amount"] = merged["amount"].where(~merged["amount_suspect"])

# (a) County summary by transaction type
county_summary = (merged
    .groupby(["county", "txn_type"])
    .agg(n_transactions=("txn_id", "count"),
         total_value=("analysis_amount", "sum"),
         avg_value=("analysis_amount", "mean"))
    .round(0))

# (b) Weekly transaction value by county (week commencing Monday)
merged["week_start"] = merged["txn_date"].dt.to_period("W").dt.start_time.dt.date
weekly_pivot = merged.pivot_table(
    index="week_start", columns="county", values="analysis_amount",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total").round(0)

# Sensitivity check: totals with and without the 250,000 values
sensitivity = pd.DataFrame({
    "total_excluding_250k": merged.groupby("county")["analysis_amount"].sum(),
    "total_including_250k": merged.groupby("county")["amount"].sum()})
sensitivity["difference"] = sensitivity["total_including_250k"] - sensitivity["total_excluding_250k"]

display(county_summary)
display(weekly_pivot)
display(sensitivity)

n_transactions  total_value  \
county                       txn_type                                  
Kiambu                       Deposit                 43      84737.0   
                             Withdrawal              27      75274.0   
Kisumu                       Deposit                 74     181091.0   
                             Withdrawal              37      74598.0   
Mombasa                      Deposit                 72     141880.0   
                             Withdrawal              61     118592.0   
Nairobi                      Deposit                118     286484.0   
                             Withdrawal              62     154930.0   
Nakuru                       Deposit                 60     181337.0   
                             Withdrawal              39      76818.0   
Unknown (unregistered agent) Deposit                  6      25279.0   
                             Withdrawal               1       1877.0   

                                         avg_value  
county                       txn_type               
Kiambu                       Deposit        2018.0  
                             Withdrawal     2895.0  
Kisumu                       Deposit        2587.0  
                             Withdrawal     2072.0  
Mombasa                      Deposit        2086.0  
                             Withdrawal     1977.0  
Nairobi                      Deposit        2535.0  
                             Withdrawal     2671.0  
Nakuru                       Deposit        3126.0  
                             Withdrawal     2076.0  
Unknown (unregistered agent) Deposit        4213.0  
                             Withdrawal     1877.0

county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown (unregistered agent),Total
week_start,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,967.0,0.0,16665.0
2026-03-02,18317.0,37933.0,33893.0,68967.0,56574.0,0.0,215684.0
2026-03-09,15995.0,23186.0,28701.0,51084.0,28730.0,0.0,147696.0
2026-03-16,22054.0,24572.0,23368.0,41331.0,39795.0,1639.0,152759.0
2026-03-23,11504.0,27077.0,17050.0,39597.0,17222.0,0.0,112450.0
2026-03-30,22609.0,41077.0,38344.0,32931.0,18145.0,8507.0,161613.0
2026-04-06,15030.0,15341.0,45316.0,55010.0,26609.0,9824.0,167130.0
2026-04-13,12265.0,25863.0,26730.0,54656.0,24376.0,0.0,143890.0
2026-04-20,24507.0,37439.0,16687.0,61634.0,30862.0,1877.0,173006.0


,total_excluding_250k,total_including_250k,difference
county,,,
Kiambu,160011.0,410011.0,250000.0
Kisumu,255689.0,505689.0,250000.0
Mombasa,260472.0,510472.0,250000.0
Nairobi,441414.0,691414.0,250000.0
Nakuru,258155.0,508155.0,250000.0
Unknown (unregistered agent),27156.0,27156.0,0.0


### Summary tables
**Method:** value totals and averages use valid amounts only. The 20 missing amounts are excluded automatically, and the 5 unverified KES 250,000 values are held out until they are checked against the ledger (question 15). Transaction counts include every transaction (600 rows). Because of this, `avg_value` is the average of known amounts and is not `total_value / n_transactions`.

**Findings:**
- **Nairobi leads** with 180 transactions and KES 441,414, followed by roughly equal Mombasa (KES 260,472), Nakuru (KES 258,155) and Kisumu (KES 255,689). Kiambu is lowest (KES 160,011). Nairobi also has the most agents (6) and Kiambu the fewest (2), so the ranking partly reflects network size.
- **Deposits outnumber withdrawals in every county**, with 373 deposits against 227 withdrawals overall. Nakuru has the largest average deposit (KES 3,126). Kiambu is the only county where the average withdrawal (KES 2,895) is clearly larger than the average deposit (KES 2,018).
- **Weekly value** peaked in the week of 2 March (KES 215,684) and was lowest in the weeks of 23 March (KES 112,450) and 27 April. The first week (23 Feb) holds only 1 March and the last week (27 April) holds only three days, so both are partial and should not be read as real dips.
- **Sensitivity:** including the five KES 250,000 values would add exactly KES 250,000 to each registered county (one in each), nearly doubling most totals and exaggerating Nairobi's lead. That is why they are excluded from the headline figures.
- 7 transactions from the unregistered agent AG099 (KES 27,156) are shown on their own line and not added to any county.

In [39]:
import pandas as pd
from cleaning_pipeline import clean_agent_data

txns = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")

clean_final, cleaning_log = clean_agent_data(txns, agents)

print(clean_final.shape)
display(cleaning_log)

# Checks against the step-by-step work
assert len(clean_final) == 600 and clean_final["txn_id"].is_unique
assert set(clean_final["txn_type"]) == {"Deposit", "Withdrawal"}
assert clean_final["amount_missing"].sum() == 20
assert clean_final["amount_suspect"].sum() == 5
assert clean_final["agent_unregistered"].sum() == 7

(600, 11)


,step,dimension,issue,rows_affected,action,rows_after
0,0,Audit,Rows loaded,612,None (starting point),612
1,1,Uniqueness,Duplicate transaction IDs,12,"Removed surplus copies, keeping the copy with ...",600
2,2,Consistency,"txn_type spelling, case and whitespace variants",376,"Stripped whitespace, lower-cased, mapped to De...",600
3,3,Validity,Dates written with '/' instead of '-',5,"Standardised separator, parsed with explicit %...",600
4,3,Validity,Dates that could not be parsed,0,Left empty (NaT),600
5,4,Completeness,"Amount missing, non-numeric ('N/A') or not pos...",20,"Kept row, set amount_missing = True, no imputa...",600
6,5,Accuracy,"IQR outliers (fences -1,928 to 6,084)",40,Flagged in is_outlier_iqr; kept in data,600
7,5,Accuracy,"Amounts of exactly KES 250,000",5,Flagged in amount_suspect; keep pending ledger...,600
8,6,Integrity,agent_id not found in the agent register,7,"Kept row, set agent_unregistered = True, count...",600
9,7,Audit,Rows returned,600,None (end point),600


### Cleaning pipeline
The cleaning steps are wrapped in `clean_agent_data(txns, agents)` in `cleaning_pipeline.py`. It returns the clean DataFrame (600 rows, one per transaction, merged with the agent register) and a cleaning log with one row per step: the dimension, the issue, the rows affected, the action taken and the rows remaining.

Design choices: the function works on a copy of its inputs, de-duplicates on `txn_id`, and flags problems (missing amounts, outliers, unregistered agents) instead of deleting rows, so nothing is lost without being recorded. Every rule is a named constant at the top of the file.

Verification: the pipeline output matches the step-by-step notebook results (12 duplicates removed, 373 deposits and 227 withdrawals, 20 missing amounts, 40 IQR outliers of which 5 are KES 250,000, 7 unregistered-agent transactions).

**DATA QUALITY REPORT**
### Privacy: personal data and the Kenya Data Protection Act 2019
The current extract holds only transaction IDs, agent IDs, transaction types, amounts and dates, which describe transactions rather than customers. If it also held **customer phone numbers**, the phone number would be personal data because it directly identifies a person. Fields that were harmless on their own (`txn_id`, `amount`, `txn_date`, `agent_id`, and through the agent the county) would also become personal data, because they could then be linked to a named individual and would reveal who sent or received money, how much, when and roughly where. Financial behaviour is not in the Act's list of "sensitive" categories as far as I know, but it should still be treated as high-risk.

**Protection measures:**
1. **Minimise:** leave phone numbers out of analytical extracts unless the analysis needs them (data minimisation and purpose limitation, section 25).
2. **Pseudonymise:** replace each number with a keyed hash (HMAC) token, with the key held separately. A plain hash of a phone number is easy to reverse, because phone numbers can be guessed.
3. **Aggregate for reporting:** county and weekly totals contain no personal data.
4. **Secure:** role-based access, encryption in transit and at rest, and an access log (data protection by design and default, section 41).
5. **Limit retention** (section 39), and be able to respond to data-subject requests for correction or erasure (sections 26 and 40).
6. **Prepare for breaches:** notify the Data Commissioner within 72 hours (section 43), and have contracts that require any processor to report breaches to us promptly.
7. **Assess:** carry out a data protection impact assessment (section 31) before processing customer-level data at scale.